In [0]:
from pyspark import pipelines as dp
import requests
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp

current_date = str(datetime.today().strftime('%Y-%m-%d'))
API_KEY = "cbcf5759b3ba91b715fd9ef2a629e678"
BASE_URL = "https://v3.football.api-sports.io"

@dp.materialized_view(
    comment="Bronze layer table containing leagues data from Football API", 
    schema="""
        league_id BIGINT NOT NULL PRIMARY KEY,
        league_name STRING,
        country STRING,
        country_code STRING,
        season_start STRING,
        season_end STRING,
        year BIGINT,
        insert_timestamp TIMESTAMP
    """

)
@dp.expect("has_league_name", "league_name IS NOT NULL AND LENGTH(league_name) > 0")

def leagues_bronze():
    headers = {
        "x-apisports-key": API_KEY
    }

    url = f"{BASE_URL}/leagues"

    response = requests.get(url, headers=headers)
    data = response.json()
    leagues_clean = []

    for item in data["response"]:
        league_id = item["league"]["id"]
        league_name = item["league"]["name"]
        country = item["country"]["name"]
        country_code = item["country"]["code"]
        for seasons in item["seasons"]:
            leagues_clean.append({
                "league_id": league_id,
                "league_name": league_name,
                "country": country,
                "country_code": country_code,
                "season_start": seasons["start"],
                "season_end": seasons["end"],
                "year": seasons["year"]
            })

    output_path = f"/Volumes/main/db_project_bronze/db_project_vol/leagues_clean{current_date}.json"

    with open(output_path, "w") as f:
        json.dump(leagues_clean, f, indent=2)

    df = spark.read.json(output_path, multiLine=True)
    df = df.withColumn("insert_timestamp", current_timestamp())

    return df

    df_time = df.withColumn("insert_timestamp", current_timestamp())
    df_time.write.mode("overwrite").saveAsTable(f"{target_db}.leauges_bronze")

    return df_time